# 🕸️ DeepGuard — Notebook 4: Graph Neural Network (Multi-GNN)
**IBM AML Dataset** | Iqra University FYP 2023 | Supervisor: Dr. Dure e Jabeen

> ⚠️ **Read this before running.** Unlike Notebooks 1–3, this notebook was **not executed and
> verified** before being handed to you — the sandbox that built it couldn't install PyTorch
> (disk-quota limits unrelated to your Colab environment). The logic follows a published,
> peer-reviewed method as closely as reasonably possible for an FYP timeline, but you are the
> first person actually running this code. Expect to debug it. If you hit an error you can't
> resolve, paste it back and we'll fix it together — that's normal for this kind of work, not a
> sign something is fundamentally wrong.
>
> **Why bother, given that:** the published benchmark this follows reports **F1 ≈ 0.71, AP ≈ 0.67**
> on this exact dataset family — a real improvement over the XGBoost result (F1 0.62, AP 0.58) in
> Notebook 3. It's also a better fit for your thesis narrative: DeepGuard visualizes transactions
> as a network graph (Cytoscape.js), so a model that actually *reasons* over that graph structure
> is more defensible than a tabular model with a graph UI bolted on.
>
> **What this implements**, based on Egressy et al., *"Provably Powerful Graph Neural Networks
> for Directed Multigraphs"* (the paper that benchmarks GNNs on this exact IBM AML dataset family):
> 1. **Reverse message passing** — each transaction edge is duplicated in reverse with a
>    direction flag, so information flows both ways along a payment (the single biggest lever in
>    the paper's ablation: +28 F1 points on its own).
> 2. **Port numbering** — an edge feature counting repeated transactions between the same pair of
>    accounts, which helps the model recognize structuring/smurfing patterns (many small transfers
>    between the same accounts).
>
> Run **Notebook 1** first, or at least have `HI-Small_Trans.csv` ready to upload below.
>
> ⚡ **Use a GPU runtime**: Runtime → Change runtime type → T4 GPU. This will be painfully slow
> on CPU.

## 📦 Step 1 — Install PyTorch Geometric

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}  CUDA available: {torch.cuda.is_available()}")

!pip install torch_geometric -q
import torch_geometric
print(f"PyTorch Geometric: {torch_geometric.__version__}")

# NOTE: this deliberately avoids the optional pyg-lib/torch-scatter/torch-sparse compiled
# extensions (they need a version-matched wheel from data.pyg.org and are a common source of
# install headaches). GINEConv, used below, does not require them.

PyTorch: 2.11.0+cu128  CUDA available: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.2 MB/s eta 0:00:00
PyTorch Geometric: 2.8.0.post1


## 📂 Step 2 — Load the raw CSV

In [ ]:
import pandas as pd, numpy as np

RAW_CSV_PATH = '/content/HI-Small_Trans.csv'   # 🔧 change if needed

# if 'google.colab' in str(get_ipython()):
#     from google.colab import files
#     print("📁 Upload HI-Small_Trans.csv")
#     uploaded = files.upload()
#     RAW_CSV_PATH = f'/content/{list(uploaded.keys())[0]}'

df = pd.read_csv(RAW_CSV_PATH)
print(f"📊 Loaded: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
print(f"🏷️  Fraud rate: {df['Is Laundering'].mean()*100:.4f}%  ({df['Is Laundering'].sum():,} / {len(df):,})")
df.head()

📊 Loaded: (5078345, 11)
📋 Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
🏷️  Fraud rate: 0.1019%  (5,177 / 5,078,345)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## 🕸️ Step 3 — Build the transaction graph

- **Nodes** = accounts (the `Account` / `Account.1` columns — already globally unique in this
  dataset, per Notebook 1's preprocessing)
- **Edges** = transactions, directed `Account → Account.1`
- **Edge features** = the same transaction-level features as Notebook 1 (amounts, time,
  currency/format), **plus** the two GNN-specific additions below
- **Node features** = simple degree/volume statistics computed from **training edges only**, to
  avoid leaking test-set information into what the model "knows" about a node

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# df = df.dropna(subset=['Timestamp', 'Amount Paid', 'Amount Received', 'Account', 'Account.1',  'Is Laundering']).reset_index(drop=True)

# df['Timestamp']  = pd.to_datetime(df['Timestamp'], errors='coerce')
# df['Hour']       = df['Timestamp'].dt.hour
# df['DayOfWeek']  = df['Timestamp'].dt.dayofweek
# df['IsWeekend']  = (df['DayOfWeek'] >= 5).astype(int)
# df['IsNightTx']  = ((df['Hour'] >= 22) | (df['Hour'] <= 5)).astype(int)

# df['Amount Paid']     = pd.to_numeric(df['Amount Paid'], errors='coerce').fillna(0)
# df['Amount Received'] = pd.to_numeric(df['Amount Received'], errors='coerce').fillna(0)
# df['Log_Amount_Paid'] = np.log1p(df['Amount Paid'])
# df['Amount_Diff']     = df['Amount Paid'] - df['Amount Received']

# le_fmt = LabelEncoder(); df['Payment Format_enc'] = le_fmt.fit_transform(df['Payment Format'].astype(str))
# le_cur = LabelEncoder(); df['Currency_enc'] = le_cur.fit_transform(df['Payment Currency'].astype(str))

# # ── Chronological ordering matters for port numbering (defined as "the Nth transaction
# # between this pair, in time order") ──
# df = df.sort_values('Timestamp').reset_index(drop=True)

# # ── Port numbering: count of prior transactions between this exact (sender, receiver) pair ──
# df['port_number'] = df.groupby(['Account', 'Account.1']).cumcount()
# print(f"✅ Port numbering: max repeated transactions between one pair = {df['port_number'].max()}")

# # ── Global account → node-index mapping ──
# all_accounts = pd.unique(pd.concat([df['Account'], df['Account.1']]))
# acct_to_idx = {acct: i for i, acct in enumerate(all_accounts)}
# n_nodes = len(all_accounts)
# print(f"✅ {n_nodes:,} unique accounts (nodes)")

# df['src'] = df['Account'].map(acct_to_idx)
# df['dst'] = df['Account.1'].map(acct_to_idx)

✅ Port numbering: max repeated transactions between one pair = 5
✅ 85,220 unique accounts (nodes)


In [ ]:
###BY CHAT
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# Clean required columns
# ─────────────────────────────────────────────

# Convert Timestamp first
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

# Convert amounts to numeric
df['Amount Paid'] = pd.to_numeric(df['Amount Paid'], errors='coerce')
df['Amount Received'] = pd.to_numeric(df['Amount Received'], errors='coerce')

# Convert target to numeric
df['Is Laundering'] = pd.to_numeric(df['Is Laundering'], errors='coerce')

# Drop rows with missing required values
df = df.dropna(subset=[
    'Timestamp',
    'Amount Paid',
    'Amount Received',
    'Account',
    'Account.1',
    'Is Laundering'
]).reset_index(drop=True)

# Target must be integer
df['Is Laundering'] = df['Is Laundering'].astype(int)

# ─────────────────────────────────────────────
# Time features
# ─────────────────────────────────────────────

df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['IsNightTx'] = ((df['Hour'] >= 22) | (df['Hour'] <= 5)).astype(int)

# ─────────────────────────────────────────────
# Amount features
# ─────────────────────────────────────────────

df['Log_Amount_Paid'] = np.log1p(df['Amount Paid'])
df['Amount_Diff'] = df['Amount Paid'] - df['Amount Received']

# ─────────────────────────────────────────────
# Encode categorical features
# ─────────────────────────────────────────────

le_fmt = LabelEncoder()
df['Payment Format_enc'] = le_fmt.fit_transform(
    df['Payment Format'].astype(str)
)

le_cur = LabelEncoder()
df['Currency_enc'] = le_cur.fit_transform(
    df['Payment Currency'].astype(str)
)

# ─────────────────────────────────────────────
# Chronological ordering
# ─────────────────────────────────────────────

df = df.sort_values('Timestamp').reset_index(drop=True)

# ─────────────────────────────────────────────
# Port numbering
# ─────────────────────────────────────────────

df['port_number'] = df.groupby(
    ['Account', 'Account.1']
).cumcount()

print(
    f"✅ Port numbering: max repeated transactions "
    f"between one pair = {df['port_number'].max()}"
)

# ─────────────────────────────────────────────
# Global account → node-index mapping
# ─────────────────────────────────────────────

all_accounts = pd.unique(
    pd.concat([df['Account'], df['Account.1']])
)

acct_to_idx = {
    acct: i for i, acct in enumerate(all_accounts)
}

n_nodes = len(all_accounts)

print(f"✅ {n_nodes:,} unique accounts (nodes)")

df['src'] = df['Account'].map(acct_to_idx)
df['dst'] = df['Account.1'].map(acct_to_idx)


# ─────────────────────────────────────────────
# Edge features
# ─────────────────────────────────────────────

EDGE_FEATURE_COLS = [
    'Amount Paid',
    'Amount Received',
    'Log_Amount_Paid',
    'Amount_Diff',
    'Hour',
    'DayOfWeek',
    'IsWeekend',
    'IsNightTx',
    'Payment Format_enc',
    'Currency_enc',
    'port_number'
]


# ─────────────────────────────────────────────
# Train / Validation / Test split
# ─────────────────────────────────────────────

edge_idx = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    edge_idx,
    test_size=0.30,
    stratify=df['Is Laundering'],
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=df.loc[temp_idx, 'Is Laundering'],
    random_state=42
)

print(
    f"Train edges: {len(train_idx):,}  "
    f"Val edges: {len(val_idx):,}  "
    f"Test edges: {len(test_idx):,}"
)

print(
    f"Fraud — train: {df.loc[train_idx, 'Is Laundering'].sum():.0f}  "
    f"val: {df.loc[val_idx, 'Is Laundering'].sum():.0f}  "
    f"test: {df.loc[test_idx, 'Is Laundering'].sum():.0f}"
)


# ─────────────────────────────────────────────
# Node features from TRAINING edges only
# ─────────────────────────────────────────────

train_df = df.loc[train_idx]

node_feat = np.zeros(
    (n_nodes, 6),
    dtype=np.float32
)

out_deg = train_df.groupby('src')['Amount Paid'].agg(
    ['count', 'sum', 'mean']
)

in_deg = train_df.groupby('dst')['Amount Received'].agg(
    ['count', 'sum', 'mean']
)

for idx, row in out_deg.iterrows():
    node_feat[idx, 0:3] = row.values

for idx, row in in_deg.iterrows():
    node_feat[idx, 3:6] = row.values

# Heavy-tailed values → log scale
node_feat = np.log1p(node_feat)

print(f"✅ Node features: {node_feat.shape}")

✅ Port numbering: max repeated transactions between one pair = 88
✅ 515,080 unique accounts (nodes)
Train edges: 3,554,841  Val edges: 761,752  Test edges: 761,752
Fraud — train: 3624  val: 777  test: 776
✅ Node features: (515080, 6)


Train edges: 2,185,165
Val edges:   468,250
Test edges:  468,250
Fraud — train: 2054  val: 441  test: 440


In [ ]:
# EDGE_FEATURE_COLS = [
#     'Amount Paid', 'Amount Received', 'Log_Amount_Paid', 'Amount_Diff',
#     'Hour', 'DayOfWeek', 'IsWeekend', 'IsNightTx',
#     'Payment Format_enc', 'Currency_enc', 'port_number'
# ]

# from sklearn.model_selection import train_test_split

# edge_idx = np.arange(len(df))
# train_idx, temp_idx = train_test_split(edge_idx, test_size=0.30, stratify=df['Is Laundering'], random_state=42)
# val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50, stratify=df.loc[temp_idx, 'Is Laundering'], random_state=42)
# print(f"Train edges: {len(train_idx):,}  Val edges: {len(val_idx):,}  Test edges: {len(test_idx):,}")
# print(f"Fraud — train: {df.loc[train_idx,'Is Laundering'].sum():.0f}  "
#       f"val: {df.loc[val_idx,'Is Laundering'].sum():.0f}  test: {df.loc[test_idx,'Is Laundering'].sum():.0f}")

# # ── Node features from TRAINING edges only (no leakage) ──
# train_df = df.loc[train_idx]
# node_feat = np.zeros((n_nodes, 6), dtype=np.float32)
# out_deg = train_df.groupby('src')['Amount Paid'].agg(['count','sum','mean'])
# in_deg  = train_df.groupby('dst')['Amount Received'].agg(['count','sum','mean'])
# for idx, row in out_deg.iterrows():
#     node_feat[idx, 0:3] = row.values
# for idx, row in in_deg.iterrows():
#     node_feat[idx, 3:6] = row.values
# node_feat = np.log1p(node_feat)  # heavy-tailed -> log scale
# print(f"✅ Node features: {node_feat.shape}")

ValueError: Input y contains NaN.

## 🔁 Step 4 — Add reverse edges (the key GNN-specific adaptation)

Standard message passing only lets information flow in the edge's stored direction
(sender → receiver). Money laundering detection needs both directions — e.g. an account is
suspicious partly because of what it does with money *after* receiving it, which a
receiver-side-only view can't see. We add a reverse copy of every edge with a `is_reverse`
flag, so a normal PyG conv layer effectively becomes bidirectional while still knowing which
direction was the real transaction.

In [ ]:
import torch
from torch_geometric.data import Data

scaler_mean = train_df[EDGE_FEATURE_COLS].mean().values
scaler_std  = train_df[EDGE_FEATURE_COLS].std().replace(0, 1).values

def scale_edges(sub_df):
    X = (sub_df[EDGE_FEATURE_COLS].values - scaler_mean) / scaler_std
    return X.astype(np.float32)

edge_feat_all = scale_edges(df)

src = df['src'].values
dst = df['dst'].values

# Forward edges (flag=0) + reverse edges (flag=1), doubling the edge list
edge_index_fwd = np.stack([src, dst])
edge_index_rev = np.stack([dst, src])
edge_index = np.concatenate([edge_index_fwd, edge_index_rev], axis=1)

flag_fwd = np.zeros((len(df), 1), dtype=np.float32)
flag_rev = np.ones((len(df), 1), dtype=np.float32)
edge_attr = np.concatenate([
    np.concatenate([edge_feat_all, flag_fwd], axis=1),
    np.concatenate([edge_feat_all, flag_rev], axis=1),
], axis=0)

y_all = df['Is Laundering'].values.astype(np.float32)
# label/masks only apply to the forward half (reverse copies are structural only, not
# separately-labeled "transactions")
n_edges = len(df)
train_mask = np.zeros(2*n_edges, dtype=bool); train_mask[train_idx] = True
val_mask   = np.zeros(2*n_edges, dtype=bool); val_mask[val_idx]     = True
test_mask  = np.zeros(2*n_edges, dtype=bool); test_mask[test_idx]   = True

data = Data(
    x=torch.tensor(node_feat, dtype=torch.float32),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    edge_attr=torch.tensor(edge_attr, dtype=torch.float32),
    y=torch.tensor(np.concatenate([y_all, y_all]), dtype=torch.float32),
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask),
)
print(data)
print(f"Edge feature dim: {data.edge_attr.shape[1]} (10 transaction features + port_number + is_reverse flag)")

Data(x=[515080, 6], edge_index=[2, 10156690], edge_attr=[10156690, 12], y=[10156690], train_mask=[10156690], val_mask=[10156690], test_mask=[10156690])
Edge feature dim: 12 (10 transaction features + port_number + is_reverse flag)


## 🧠 Step 5 — Model: GINE-based Multi-GNN with edge classification head

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv

class MultiGNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden=64, n_layers=3, dropout=0.2):
        super().__init__()
        self.node_proj = nn.Linear(node_in, hidden)
        self.edge_proj = nn.Linear(edge_in, hidden)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

        # Edge classifier: [h_src, h_dst, edge_embedding] -> logit
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.node_proj(x))
        e = F.relu(self.edge_proj(edge_attr))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, e)
            h_new = norm(h_new)
            h = F.relu(h_new) + h  # residual
            h = F.dropout(h, p=self.dropout, training=self.training)

        src, dst = edge_index
        edge_logits = self.edge_mlp(torch.cat([h[src], h[dst], e], dim=1))
        return edge_logits.squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiGNN(node_in=data.x.shape[1], edge_in=data.edge_attr.shape[1]).to(device)
data = data.to(device)
print(model)
print(f"Device: {device}")

MultiGNN(
  (node_proj): Linear(in_features=6, out_features=64, bias=True)
  (edge_proj): Linear(in_features=12, out_features=64, bias=True)
  (convs): ModuleList(
    (0-2): 3 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (norms): ModuleList(
    (0-2): 3 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=192, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
Device: cuda


## 🏋️ Step 6 — Train

Full-batch (the whole graph fits in memory for `HI-Small` on a T4 — a few hundred thousand
nodes, a few million edges). **If you get a CUDA out-of-memory error**: reduce `hidden` to 32,
or switch to PyG's `NeighborLoader` for mini-batch training (a bigger change — ask for help if
you need to go this route).

Loss uses `pos_weight` the same way we used `scale_pos_weight` for XGBoost, and — same
discipline as every other notebook here — the decision threshold is chosen on the **validation**
edges only, evaluated on test **once**.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score, classification_report, confusion_matrix

n_pos = data.y[data.train_mask].sum()
n_neg = data.train_mask.sum() - n_pos
pos_weight = torch.tensor([(n_neg / n_pos).item()], device=device)
print(f"pos_weight = {pos_weight.item():.1f}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_ap = 0.0
best_state = None
patience, patience_ctr = 15, 0
EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        val_scores = torch.sigmoid(logits[data.val_mask]).cpu().numpy()
        val_y = data.y[data.val_mask].cpu().numpy()
        val_ap = average_precision_score(val_y, val_scores)

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  loss={loss.item():.4f}  val_AP={val_ap:.4f}  (best={best_val_ap:.4f})")

    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch} (no val_AP improvement for {patience} epochs)")
        break

model.load_state_dict(best_state)
print(f"\n✅ Best validation AP: {best_val_ap:.4f}")

pos_weight = 979.9


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.42 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.30 GiB is free. Including non-PyTorch memory, this process has 13.26 GiB memory in use. Of the allocated memory 11.27 GiB is allocated by PyTorch, and 1.86 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
from torch_geometric.loader import NeighborLoader

data = data.cpu()                      # keep the full graph off the GPU
NUM_NEIGHBORS = [15, 10]               # one entry per GNN layer in your model

train_loader = NeighborLoader(
    data, num_neighbors=NUM_NEIGHBORS, batch_size=4096,
    input_nodes=data.train_mask, shuffle=True, num_workers=2,
)
val_loader = NeighborLoader(
    data, num_neighbors=NUM_NEIGHBORS, batch_size=8192,
    input_nodes=data.val_mask, shuffle=False, num_workers=2,
)

n_pos = data.y[data.train_mask].sum()
n_neg = data.train_mask.sum() - n_pos
pos_weight = torch.tensor([(n_neg / n_pos).item()], device=device)
print(f"pos_weight = {pos_weight.item():.1f}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_ap, best_state = 0.0, None
patience, patience_ctr = 15, 0
EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_n = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        bs = batch.batch_size                      # seed nodes come first in the batch
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(logits[:bs], batch.y[:bs])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bs
        total_n += bs

    model.eval()
    scores, ys = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            bs = batch.batch_size
            out = model(batch.x, batch.edge_index, batch.edge_attr)[:bs]
            scores.append(torch.sigmoid(out).cpu())
            ys.append(batch.y[:bs].cpu())
    val_scores = torch.cat(scores).numpy()
    val_y = torch.cat(ys).numpy()
    val_ap = average_precision_score(val_y, val_scores)

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  loss={total_loss/total_n:.4f}  val_AP={val_ap:.4f}  (best={best_val_ap:.4f})")

    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\n✅ Best validation AP: {best_val_ap:.4f}")

/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


pos_weight = 979.9


ImportError: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/node_loader.py", line 147, in collate_fn
    out = self.node_sampler.sample_from_nodes(input_data)
  File "/usr/local/lib/python3.13/dist-packages/torch_geometric/sampler/neighbor_sampler.py", line 403, in sample_from_nodes
    out = node_sample(inputs, self._sample)
  File "/usr/local/lib/python3.13/dist-packages/torch_geometric/sampler/neighbor_sampler.py", line 815, in node_sample
    out = sample_fn(seed, seed_time)
  File "/usr/local/lib/python3.13/dist-packages/torc

## 📊 Step 7 — Final evaluation (test set, touched once)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    all_scores = torch.sigmoid(logits).cpu().numpy()
    all_y = data.y.cpu().numpy()

val_scores  = all_scores[data.val_mask.cpu().numpy()]
val_y       = all_y[data.val_mask.cpu().numpy()]
test_scores = all_scores[data.test_mask.cpu().numpy()]
test_y      = all_y[data.test_mask.cpu().numpy()]

# threshold chosen on VALIDATION only
prec_v, rec_v, thr_v = precision_recall_curve(val_y, val_scores)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
best_thr = thr_v[np.argmax(f1_v)] if len(f1_v) else 0.5
print(f"Validation-chosen threshold: {best_thr:.4f}  (val F1={f1_v.max():.4f})")

test_roc = roc_auc_score(test_y, test_scores)
test_ap  = average_precision_score(test_y, test_scores)
test_pred = (test_scores >= best_thr).astype(int)
test_f1  = f1_score(test_y, test_pred)

print(f"\n=== GNN — FINAL TEST RESULTS ===")
print(f"ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print(classification_report(test_y, test_pred, target_names=['Normal','Fraud'], digits=4))
print(confusion_matrix(test_y, test_pred))

print(f"\n📊 Comparison:")
print(f"  XGBoost (Notebook 3):  ROC-AUC=0.9838  AP=0.5766  F1=0.6207")
print(f"  GNN (this notebook):   ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print(f"  Published Multi-PNA+EU on this data family: ROC-AUC~0.982-0.986  AP=0.672  F1=0.709")

## 💾 Step 8 — Save the model

If the GNN beats XGBoost, use it as your headline result and keep XGBoost as a documented
baseline/comparison in your report. If it doesn't (quite possible on a first pass — see the
honesty note in Step 9), keep XGBoost as the deployed model and report the GNN attempt with its
real numbers; a well-explained negative result is still a legitimate FYP contribution.

In [ ]:
import os, json
MODEL_PATH = '/content/saved_models'
os.makedirs(MODEL_PATH, exist_ok=True)

torch.save(model.state_dict(), f'{MODEL_PATH}/gnn_model.pt')
np.save(f'{MODEL_PATH}/gnn_scaler_mean.npy', scaler_mean)
np.save(f'{MODEL_PATH}/gnn_scaler_std.npy', scaler_std)

gnn_metadata = {
    'architecture': 'GINE-based Multi-GNN with reverse message passing + port numbering',
    'edge_feature_cols': EDGE_FEATURE_COLS + ['is_reverse'],
    'hidden_dim': 64, 'n_layers': 3,
    'threshold': float(best_thr),
    'test_performance': {'roc_auc': float(test_roc), 'avg_prec': float(test_ap), 'f1': float(test_f1)},
    'comparison': {
        'xgboost_roc_auc': 0.9838, 'xgboost_avg_prec': 0.5766, 'xgboost_f1': 0.6207,
        'published_multipna_roc_auc_range': [0.982, 0.986],
        'published_multipna_avg_prec': 0.672, 'published_multipna_f1': 0.709,
    }
}
with open(f'{MODEL_PATH}/gnn_metadata.json', 'w') as f:
    json.dump(gnn_metadata, f, indent=2)

print("✅ Saved gnn_model.pt, gnn_metadata.json")
print(json.dumps(gnn_metadata['test_performance'], indent=2))

import shutil
shutil.make_archive('/content/deepguard_gnn_model', 'zip', MODEL_PATH)
try:
    from google.colab import files
    files.download('/content/deepguard_gnn_model.zip')
except ImportError:
    print("ℹ️  Not in Colab — find the zip at /content/deepguard_gnn_model.zip")

## ⚠️ Step 9 — Reading your results honestly

A few things that can happen on a first run, and what they'd mean:

- **GNN beats XGBoost (F1 > 0.62, AP > 0.58):** Great — matches the published pattern. Use this
  as your primary model, cite the ~10–15 point AP/F1 gap over tabular XGBoost as your key
  finding, and explain *why* (graph structure captures multi-hop laundering patterns a
  per-transaction model can't see).
- **GNN is close to or slightly below XGBoost:** Also a legitimate, explainable result — not a
  failure. The published paper's full gain required more adaptations than reverse-MP + port
  numbering alone (see the paper's ablation table); this notebook implements the two adaptations
  shown to matter most, not the complete architecture. Report both numbers, explain the gap
  honestly, and you still have the stronger deployed model (XGBoost) either way.
- **GNN is dramatically worse (F1 near 0, or predicts everything as one class):** Usually means
  training instability, not a fundamentally broken idea. Try: a smaller learning rate (1e-4), a
  smaller `hidden` dim, or fewer layers (2 instead of 3) before concluding it doesn't work.
- **Training or inference crashes with an OOM error:** The full graph didn't fit on the GPU.
  Reduce `hidden` to 32, or come back for a mini-batch (`NeighborLoader`) version.

Whatever happens, **do not report a number you haven't actually seen come out of this notebook.**
If something looks off, paste the output back and we'll debug it together — that's a much
stronger position for your defense than a number you can't explain if asked how it was produced.

#############nnew

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}   CUDA available: {torch.cuda.is_available()}")

!pip install -q torch_geometric
import torch_geometric
print(f"PyTorch Geometric: {torch_geometric.__version__}")

# Only GINEConv is used from PyG (pure Python). No pyg-lib / torch-scatter / torch-sparse needed.

PyTorch: 2.11.0+cu128   CUDA available: True
PyTorch Geometric: 2.8.0.post1


In [ ]:
import os, json, time, gc
import numpy as np
import pandas as pd
import torch

# ── Data ──
RAW_CSV_PATH = '/content/HI-Small_Trans.csv'   # 🔧 change if needed
NROWS        = None      # e.g. 500_000 for a quick smoke test; None = full file
SEED         = 42

# ── Model / sampling ──
HIDDEN     = 64
FANOUTS    = [20, 10]    # neighbours sampled per node at each hop; len(FANOUTS) = number of GNN layers
N_LAYERS   = len(FANOUTS)
DROPOUT    = 0.2

# ── Training ──
TRAIN_BS   = 2048        # seed transactions per training step
EVAL_BS    = 4096        # seed transactions per inference step
NEG_RATIO  = 20          # each epoch: all fraud edges + NEG_RATIO x that many random normal edges
POS_WEIGHT = 1.0
LR         = 1e-3
EPOCHS     = 60
PATIENCE   = 10
VAL_NEG_SAMPLE = 100_000 # normal validation edges used for per-epoch early stopping

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
if device.type == 'cpu':
    print("⚠️  No GPU detected — this will be very slow. Runtime → Change runtime type → T4 GPU.")

torch.manual_seed(SEED); np.random.seed(SEED)

Device: cuda


In [ ]:
if not os.path.exists(RAW_CSV_PATH):
    try:
        from google.colab import files
        print("📁 Upload HI-Small_Trans.csv")
        uploaded = files.upload()
        RAW_CSV_PATH = '/content/' + list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{RAW_CSV_PATH} not found")

df = pd.read_csv(RAW_CSV_PATH, nrows=NROWS)
required = ['Timestamp', 'Account', 'Account.1', 'Amount Received', 'Amount Paid',
            'Payment Currency', 'Payment Format', 'Is Laundering']
missing = [c for c in required if c not in df.columns]
assert not missing, f"CSV is missing columns: {missing}. Found: {list(df.columns)}"

print(f"📊 Loaded: {df.shape}")
print(f"🏷️  Fraud rate: {df['Is Laundering'].mean()*100:.4f}%  ({df['Is Laundering'].sum():,} / {len(df):,})")
df.head()

📊 Loaded: (5078345, 11)
🏷️  Fraud rate: 0.1019%  (5,177 / 5,078,345)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [ ]:
# ── Parse / clean ──
ts = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M', errors='coerce')
if ts.isna().mean() > 0.5:                      # format guess was wrong -> let pandas infer
    ts = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Timestamp']       = ts
df['Amount Paid']     = pd.to_numeric(df['Amount Paid'], errors='coerce')
df['Amount Received'] = pd.to_numeric(df['Amount Received'], errors='coerce')
df['Is Laundering']   = pd.to_numeric(df['Is Laundering'], errors='coerce')

df = df.dropna(subset=['Timestamp', 'Amount Paid', 'Amount Received',
                       'Account', 'Account.1', 'Is Laundering']).reset_index(drop=True)
df['Is Laundering'] = df['Is Laundering'].astype(np.int64)
df = df.sort_values('Timestamp', kind='stable').reset_index(drop=True)   # chronological (needed for port numbers)
N = len(df)
print(f"✅ {N:,} transactions after cleaning")

# ── Node ids ──
if 'From Bank' in df.columns and 'To Bank' in df.columns:
    src_key = df['From Bank'].astype(str) + '_' + df['Account'].astype(str)
    dst_key = df['To Bank'].astype(str)   + '_' + df['Account.1'].astype(str)
else:
    src_key = df['Account'].astype(str)
    dst_key = df['Account.1'].astype(str)
codes, uniques = pd.factorize(pd.concat([src_key, dst_key], ignore_index=True))
src = codes[:N].astype(np.int64)
dst = codes[N:].astype(np.int64)
n_nodes = len(uniques)
del src_key, dst_key, codes, uniques; gc.collect()
print(f"✅ {n_nodes:,} unique accounts (nodes)")

# ── Port numbering ──
port_number = pd.DataFrame({'s': src, 'd': dst}).groupby(['s', 'd']).cumcount().values
print(f"✅ Port numbering: max repeated transactions between one pair = {port_number.max()}")

# ── Numeric edge features (log-scaled: amounts are extremely heavy-tailed) ──
amt_paid = df['Amount Paid'].clip(lower=0).values
amt_recv = df['Amount Received'].clip(lower=0).values
diff     = amt_paid - amt_recv
hour     = df['Timestamp'].dt.hour.values
dow      = df['Timestamp'].dt.dayofweek.values

NUM_COLS = ['Log_Amount_Paid', 'Log_Amount_Received', 'SignedLog_Amount_Diff',
            'Hour', 'DayOfWeek', 'IsWeekend', 'IsNightTx', 'Log_Port_Number']
num_raw = np.column_stack([
    np.log1p(amt_paid), np.log1p(amt_recv), np.sign(diff) * np.log1p(np.abs(diff)),
    hour, dow, (dow >= 5), ((hour >= 22) | (hour <= 5)), np.log1p(port_number),
]).astype(np.float32)

# ── Categorical features -> one-hot ──
cat_df   = pd.get_dummies(df[['Payment Format', 'Payment Currency']].astype(str), dtype=np.float32)
CAT_COLS = list(cat_df.columns)
cat_arr  = cat_df.values.astype(np.float32)
del cat_df

EDGE_FEATURE_COLS = NUM_COLS + CAT_COLS
y     = df['Is Laundering'].values.astype(np.float32)
y_int = df['Is Laundering'].values.astype(np.int64)
print(f"✅ Edge features: {len(NUM_COLS)} numeric + {len(CAT_COLS)} one-hot = {len(EDGE_FEATURE_COLS)}  (+1 is_reverse flag added later)")

✅ 5,078,345 transactions after cleaning
✅ 515,088 unique accounts (nodes)
✅ Port numbering: max repeated transactions between one pair = 88
✅ Edge features: 8 numeric + 22 one-hot = 30  (+1 is_reverse flag added later)


In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(N)
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=y_int, random_state=SEED)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50, stratify=y_int[temp_idx], random_state=SEED)
print(f"Train edges: {len(train_idx):,}  Val edges: {len(val_idx):,}  Test edges: {len(test_idx):,}")
print(f"Fraud — train: {int(y[train_idx].sum())}  val: {int(y[val_idx].sum())}  test: {int(y[test_idx].sum())}")

# ── Edge feature scaling (train stats) ──
num_mean = num_raw[train_idx].mean(0)
num_std  = num_raw[train_idx].std(0); num_std[num_std == 0] = 1.0
num_scaled = np.clip((num_raw - num_mean) / num_std, -10, 10).astype(np.float32)
edge_feat  = np.hstack([num_scaled, cat_arr]).astype(np.float32)        # [N, F]
del num_raw, num_scaled, cat_arr; gc.collect()

# ── Node features from TRAINING edges only ──
tr = df.iloc[train_idx].assign(src=src[train_idx], dst=dst[train_idx])
node_feat = np.zeros((n_nodes, 6), dtype=np.float32)
og = tr.groupby('src')['Amount Paid'].agg(['count', 'sum', 'mean'])
ig = tr.groupby('dst')['Amount Received'].agg(['count', 'sum', 'mean'])
node_feat[og.index.values, 0:3] = og.values
node_feat[ig.index.values, 3:6] = ig.values
node_feat = np.log1p(node_feat)
nf_mean = node_feat.mean(0)
nf_std  = node_feat.std(0) + 1e-6
node_feat = ((node_feat - nf_mean) / nf_std).astype(np.float32)
del tr, og, ig; gc.collect()

print(f"✅ edge_feat {edge_feat.shape}   node_feat {node_feat.shape}")

Train edges: 3,554,841  Val edges: 761,752  Test edges: 761,752
Fraud — train: 3624  val: 777  test: 776
✅ edge_feat (5078345, 30)   node_feat (515088, 6)


In [ ]:
E = N
src_g       = torch.from_numpy(src).to(device)             # [E]  forward sender
dst_g       = torch.from_numpy(dst).to(device)             # [E]  forward receiver
y_g         = torch.from_numpy(y).to(device)               # [E]
edge_feat_g = torch.from_numpy(edge_feat).to(device)       # [E, F]
node_x_g    = torch.from_numpy(node_feat).to(device)       # [n_nodes, 6]

# all 2E directed message edges: [0,E) forward, [E,2E) reverse
m_src = torch.cat([src_g, dst_g])
m_dst = torch.cat([dst_g, src_g])
order    = torch.argsort(m_dst, stable=True)               # group by target node  -> CSR
csr_eid  = order                                           # message-edge id at each CSR position
csr_src  = m_src[order]
csr_dst  = m_dst[order]
indptr   = torch.zeros(n_nodes + 1, dtype=torch.long, device=device)
indptr[1:] = torch.cumsum(torch.bincount(m_dst, minlength=n_nodes), 0)
remap    = torch.zeros(n_nodes, dtype=torch.long, device=device)   # global node id -> local id (scratch)
del m_src, m_dst, order
if device.type == 'cuda':
    torch.cuda.empty_cache()
    print(f"GPU memory used by graph: {torch.cuda.memory_allocated()/2**30:.2f} GiB")

def _sample_hop(targets, fanout):
    # For every target node pick up to `fanout` incoming edges (all of them if degree <= fanout).
    # Returns positions into the CSR arrays.
    start = indptr[targets]
    deg   = indptr[targets + 1] - start
    cnt   = torch.clamp(deg, max=fanout)
    total = int(cnt.sum().item())
    if total == 0:
        return torch.empty(0, dtype=torch.long, device=device)
    owner = torch.repeat_interleave(torch.arange(len(targets), device=device), cnt)
    first = torch.cumsum(cnt, 0) - cnt
    j     = torch.arange(total, device=device) - first[owner]          # 0..cnt-1 inside each target
    d     = deg[owner]
    rnd   = torch.minimum((torch.rand(total, device=device) * d).long(), d - 1)
    off   = torch.where(d <= fanout, j, rnd)                            # take all, or random draw
    return start[owner] + off

def build_batch(seed_eids):
    # seed_eids: forward-transaction ids (0..E-1). Returns (model inputs dict, labels).
    s_src, s_dst = src_g[seed_eids], dst_g[seed_eids]
    frontier  = torch.unique(torch.cat([s_src, s_dst]))
    seed_nodes = frontier
    chunks = []
    for fo in FANOUTS:
        pos = _sample_hop(frontier, fo)
        chunks.append(pos)
        frontier = torch.unique(csr_src[pos])
    pos   = torch.unique(torch.cat(chunks))                             # de-duplicate sampled edges
    e_src, e_dst, e_id = csr_src[pos], csr_dst[pos], csr_eid[pos]

    nodes = torch.unique(torch.cat([seed_nodes, e_src, e_dst]))
    remap[nodes] = torch.arange(len(nodes), device=device)

    is_rev    = (e_id >= E).float().unsqueeze(1)
    edge_attr = torch.cat([edge_feat_g[e_id % E], is_rev], dim=1)
    seed_attr = torch.cat([edge_feat_g[seed_eids],
                           torch.zeros(len(seed_eids), 1, device=device)], dim=1)
    inputs = dict(
        x=node_x_g[nodes],
        edge_index=torch.stack([remap[e_src], remap[e_dst]]),
        edge_attr=edge_attr,
        seed_src=remap[s_src], seed_dst=remap[s_dst], seed_attr=seed_attr,
    )
    return inputs, y_g[seed_eids]

# quick sanity check
_inp, _y = build_batch(torch.from_numpy(train_idx[:TRAIN_BS]).to(device))
print(f"Example batch -> nodes: {_inp['x'].shape[0]:,}  edges: {_inp['edge_index'].shape[1]:,}  seeds: {len(_y):,}")
del _inp, _y

GPU memory used by graph: 0.92 GiB
Example batch -> nodes: 35,759  edges: 158,707  seeds: 2,048


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv

class MultiGNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.node_proj = nn.Linear(node_in, hidden)
        self.edge_proj = nn.Linear(edge_in, hidden)
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, edge_index, edge_attr, seed_src, seed_dst, seed_attr):
        h = F.relu(self.node_proj(x))
        e = F.relu(self.edge_proj(edge_attr))
        for conv, norm in zip(self.convs, self.norms):
            h = h + F.relu(norm(conv(h, edge_index, e)))        # residual
            h = F.dropout(h, p=self.dropout, training=self.training)
        e_seed = F.relu(self.edge_proj(seed_attr))              # the transaction being classified
        z = torch.cat([h[seed_src], h[seed_dst], e_seed], dim=1)
        return self.edge_mlp(z).squeeze(-1)

model = MultiGNN(node_in=node_feat.shape[1], edge_in=edge_feat.shape[1] + 1,
                 hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

MultiGNN(
  (node_proj): Linear(in_features=6, out_features=64, bias=True)
  (edge_proj): Linear(in_features=31, out_features=64, bias=True)
  (convs): ModuleList(
    (0-1): 2 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (norms): ModuleList(
    (0-1): 2 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=192, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
Parameters: 40,129


In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             f1_score, classification_report, confusion_matrix)

train_t = torch.from_numpy(train_idx).to(device)
val_t   = torch.from_numpy(val_idx).to(device)
test_t  = torch.from_numpy(test_idx).to(device)

tr_pos = train_t[y_g[train_t] == 1]
tr_neg = train_t[y_g[train_t] == 0]
n_neg_epoch = min(len(tr_neg), NEG_RATIO * len(tr_pos))
print(f"Train fraud edges: {len(tr_pos):,}   normal edges: {len(tr_neg):,}   -> per epoch: {len(tr_pos):,} + {n_neg_epoch:,}")

# fixed validation subsample (+ weights that undo the negative subsampling)
va_pos = val_t[y_g[val_t] == 1]
va_neg = val_t[y_g[val_t] == 0]
g   = torch.Generator().manual_seed(SEED)
sel = torch.randperm(len(va_neg), generator=g)[:VAL_NEG_SAMPLE].to(device)
va_sub   = torch.cat([va_pos, va_neg[sel]])
va_sub_y = y_g[va_sub].cpu().numpy()
va_sub_w = np.where(va_sub_y == 1, 1.0, len(va_neg) / max(len(sel), 1)).astype(np.float64)

@torch.no_grad()
def predict(eids, bs=EVAL_BS, seed=0):
    # Probability of fraud for each transaction id in `eids` (deterministic neighbour sampling).
    model.eval()
    outs = []
    rng_devices = [device.index or 0] if device.type == 'cuda' else []
    with torch.random.fork_rng(devices=rng_devices):
        torch.manual_seed(seed)
        for i in range(0, len(eids), bs):
            inputs, _ = build_batch(eids[i:i + bs])
            outs.append(torch.sigmoid(model(**inputs)).cpu())
    return torch.cat(outs).numpy()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))

best_val_ap, best_state, bad_epochs = -1.0, None, 0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    neg   = tr_neg[torch.randperm(len(tr_neg), device=device)[:n_neg_epoch]]
    seeds = torch.cat([tr_pos, neg])
    seeds = seeds[torch.randperm(len(seeds), device=device)]

    tot_loss, tot_n = 0.0, 0
    for i in range(0, len(seeds), TRAIN_BS):
        inputs, yb = build_batch(seeds[i:i + TRAIN_BS])
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(**inputs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        tot_loss += loss.item() * len(yb); tot_n += len(yb)

    val_scores = predict(va_sub)
    val_ap = average_precision_score(va_sub_y, val_scores, sample_weight=va_sub_w)
    history.append((epoch, tot_loss / tot_n, val_ap))

    if val_ap > best_val_ap:
        best_val_ap, bad_epochs = val_ap, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1

    print(f"Epoch {epoch:3d}  loss={tot_loss/tot_n:.4f}  val_AP={val_ap:.4f}  (best={best_val_ap:.4f})  [{time.time()-t0:.0f}s]")
    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val_AP improvement for {PATIENCE} epochs)")
        break

model.load_state_dict(best_state)
print(f"\n✅ Best validation AP (subsample, re-weighted): {best_val_ap:.4f}")

Train fraud edges: 3,624   normal edges: 3,551,217   -> per epoch: 3,624 + 72,480
Epoch   1  loss=0.2168  val_AP=0.0143  (best=0.0143)  [2s]
Epoch   2  loss=0.1364  val_AP=0.0520  (best=0.0520)  [1s]
Epoch   3  loss=0.1040  val_AP=0.1758  (best=0.1758)  [1s]
Epoch   4  loss=0.0802  val_AP=0.2878  (best=0.2878)  [1s]
Epoch   5  loss=0.0674  val_AP=0.3180  (best=0.3180)  [1s]
Epoch   6  loss=0.0628  val_AP=0.3135  (best=0.3180)  [1s]
Epoch   7  loss=0.0625  val_AP=0.3870  (best=0.3870)  [1s]
Epoch   8  loss=0.0595  val_AP=0.4078  (best=0.4078)  [1s]
Epoch   9  loss=0.0582  val_AP=0.4063  (best=0.4078)  [1s]
Epoch  10  loss=0.0560  val_AP=0.4207  (best=0.4207)  [1s]
Epoch  11  loss=0.0556  val_AP=0.4388  (best=0.4388)  [1s]
Epoch  12  loss=0.0546  val_AP=0.4735  (best=0.4735)  [1s]
Epoch  13  loss=0.0535  val_AP=0.4532  (best=0.4735)  [1s]
Epoch  14  loss=0.0525  val_AP=0.4746  (best=0.4746)  [1s]
Epoch  15  loss=0.0511  val_AP=0.4812  (best=0.4812)  [1s]
Epoch  16  loss=0.0521  val_AP=0.

In [ ]:
val_scores  = predict(val_t)
test_scores = predict(test_t)
val_y  = y_int[val_idx]
test_y = y_int[test_idx]

# threshold chosen on VALIDATION only
prec_v, rec_v, thr_v = precision_recall_curve(val_y, val_scores)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
best_thr = float(thr_v[np.argmax(f1_v)]) if len(f1_v) else 0.5
print(f"Validation-chosen threshold: {best_thr:.4f}  (val F1={f1_v.max():.4f}, val AP={average_precision_score(val_y, val_scores):.4f})")

test_roc  = roc_auc_score(test_y, test_scores)
test_ap   = average_precision_score(test_y, test_scores)
test_pred = (test_scores >= best_thr).astype(int)
test_f1   = f1_score(test_y, test_pred)

print("\n=== GNN — FINAL TEST RESULTS ===")
print(f"ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print(classification_report(test_y, test_pred, target_names=['Normal', 'Fraud'], digits=4))
print(confusion_matrix(test_y, test_pred))

print("\n📊 Comparison:")
print("  XGBoost (Notebook 3):  ROC-AUC=0.9838  AP=0.5766  F1=0.6207")
print(f"  GNN (this notebook):   ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print("  Published Multi-PNA+EU on this data family: ROC-AUC~0.982-0.986  AP=0.672  F1=0.709")

Validation-chosen threshold: 0.9329  (val F1=0.5663, val AP=0.5330)

=== GNN — FINAL TEST RESULTS ===
ROC-AUC=0.9882  AP=0.5562  F1=0.5637
              precision    recall  f1-score   support

      Normal     0.9995    0.9998    0.9996    760976
       Fraud     0.6738    0.4845    0.5637       776

    accuracy                         0.9992    761752
   macro avg     0.8367    0.7421    0.7817    761752
weighted avg     0.9991    0.9992    0.9992    761752

[[760794    182]
 [   400    376]]

📊 Comparison:
  XGBoost (Notebook 3):  ROC-AUC=0.9838  AP=0.5766  F1=0.6207
  GNN (this notebook):   ROC-AUC=0.9882  AP=0.5562  F1=0.5637
  Published Multi-PNA+EU on this data family: ROC-AUC~0.982-0.986  AP=0.672  F1=0.709


In [ ]:
MODEL_PATH = '/content/saved_models'
os.makedirs(MODEL_PATH, exist_ok=True)

torch.save(model.state_dict(), f'{MODEL_PATH}/gnn_model.pt')
np.save(f'{MODEL_PATH}/gnn_edge_num_mean.npy', num_mean)
np.save(f'{MODEL_PATH}/gnn_edge_num_std.npy',  num_std)
np.save(f'{MODEL_PATH}/gnn_node_feat_mean.npy', nf_mean)
np.save(f'{MODEL_PATH}/gnn_node_feat_std.npy',  nf_std)

gnn_metadata = {
    'architecture': 'GINE Multi-GNN, reverse message passing + port numbering, mini-batch neighbour sampling',
    'edge_numeric_cols': NUM_COLS,
    'edge_onehot_cols': CAT_COLS,
    'edge_extra_flag': 'is_reverse (last column)',
    'hidden_dim': HIDDEN, 'n_layers': N_LAYERS, 'fanouts': FANOUTS, 'dropout': DROPOUT,
    'threshold': best_thr,
    'test_performance': {'roc_auc': float(test_roc), 'avg_prec': float(test_ap), 'f1': float(test_f1)},
    'comparison': {
        'xgboost_roc_auc': 0.9838, 'xgboost_avg_prec': 0.5766, 'xgboost_f1': 0.6207,
        'published_multipna_roc_auc_range': [0.982, 0.986],
        'published_multipna_avg_prec': 0.672, 'published_multipna_f1': 0.709,
    },
}
with open(f'{MODEL_PATH}/gnn_metadata.json', 'w') as f:
    json.dump(gnn_metadata, f, indent=2)

print("✅ Saved model + scalers + metadata")
print(json.dumps(gnn_metadata['test_performance'], indent=2))

import shutil
shutil.make_archive('/content/deepguard_gnn_model', 'zip', MODEL_PATH)
try:
    from google.colab import files
    files.download('/content/deepguard_gnn_model.zip')
except ImportError:
    print("ℹ️  Not in Colab — find the zip at /content/deepguard_gnn_model.zip")

✅ Saved model + scalers + metadata
{
  "roc_auc": 0.988163451603497,
  "avg_prec": 0.5561571667281828,
  "f1": 0.5637181409295352
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>